# Trip-chain browser

Walk through the persons whose trip chain carries a given `problemas` code (a defect left) or `ajustes`
code (a change `load_eod` made), one person at a time, with the chain **as shipped** next to what
**`load_eod()` returns**. Every table is read live through `load_eod`, so the browser reflects the current
rules after any change to `eodgdl.chains`. The table is `eodgdl.review.chain_sheet`, the one
`eodgdl review export` writes as a CSV, so what is browsed here and what is edited there are the same rows.

**How to use.** Pick codes in the two lists (Ctrl/Cmd-click for several; `dropped` selects the persons who
lost a duplicate return). *any selected* keeps a person if one of their rows carries any chosen code;
*all selected* keeps a person only if their chain carries every chosen code. Step with **◀ ▶**, jump with
the counter or **random**, or type `household,person` in *go to*. Rows that carry a selected code are
highlighted; a cell that the rules changed reads `shipped → cleaned`; a dropped row is struck through.
In the timeline, grey bars are returns home, blue bars activities, each running from the start time
across the leg minutes; a hatched bar was edited or imputed, a red edge carries a `problemas` code.

**Review sheet.** `eodgdl review export --data data --out notebooks/chain_review.csv` writes the chains
pending a fix in this table's format, with an empty `new …` column next to every column that may take a
change (`new start`, `new motive`, `new dest. type`, `new orig. type`, `new origin`, `new destination`,
`new mode`, `new status`). Write the value that should hold there, never in the original column:
`dropped` under *new status* takes a row out, `restored` brings back a row `load_eod` dropped, and *note*
says why. Put the file's path in *review sheet* and press **load sheet** (`notebooks/chain_review.csv`
loads on start): the filled cells are the edits, they are applied to the live tables and `problemas` is
recomputed, so every edited person gets a table **after your edits** with the defects the edits cleared,
left or made, and a third timeline lane; *edited only* filters to them. An edit that cannot be applied (an
unknown label or zone, a bad time) is reported instead, and nothing is applied.

Notes typed under a person are saved to `notebooks/chain_browser_notes.json` (or the working directory
when `EODGDL_DATA_DIR` points elsewhere); *with a note only* filters to them, so a pattern can be
collected across sessions.

In [ ]:
import json
import os
import random
from pathlib import Path

import ipywidgets as w
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, display


def _find_data_dir() -> Path:
    marker = "IMEPLAN_Base_Viviendas_Master.csv"
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / "data"
        if (cand / marker).exists():
            return cand.resolve()
    raise FileNotFoundError("Could not locate the eodgdl data/ directory.")


os.environ.setdefault("EODGDL_DATA_DIR", str(_find_data_dir()))

from eodgdl import load_eod, review
from eodgdl.chains import HOME_MOTIVE

P = ["folio_vivienda", "folio_habitante"]
DATA_DIR = Path(os.environ["EODGDL_DATA_DIR"])
NOTES_DIR = DATA_DIR.parent / "notebooks" if (DATA_DIR.parent / "notebooks").is_dir() else Path.cwd()
NOTES_PATH = NOTES_DIR / "chain_browser_notes.json"
REVIEW_PATH = NOTES_DIR / "chain_review.csv"      # the review sheet, once exported and edited (eodgdl review export)

shipped = load_eod(clean_chains=False)
cleaned = load_eod()
rows = review.chain_rows(cleaned, shipped)        # one row per shipped trip: shipped and cleaned values, codes, dropped, home
codes = review.code_columns(rows)                 # one boolean column per code, computed once so the filters are instant


def codes_of(s):
    return frozenset(c for v in s for c in v.split(";") if c)


by_person = rows.groupby(level=P)
persons = pd.DataFrame({
    "n_trips": by_person.size(),
    "fix_codes": by_person.ajustes.agg(codes_of),
    "issue_codes": by_person.problemas.agg(codes_of),
    "dropped": by_person.dropped.any(),
}).join(cleaned.hab[["sexo_nacimiento", "edad", "ocupacion", "viajes_contados", "diario_repetido"]])
persons["home"] = by_person.home.first()
person_codes = codes.groupby(level=P).any()
ISSUE_OPTIONS = [c for c in codes.columns if c in review.ISSUE_CODES] + ["dropped"]
FIX_OPTIONS = [c for c in codes.columns if c not in ISSUE_OPTIONS]

notes = json.loads(NOTES_PATH.read_text()) if NOTES_PATH.exists() else {}

summary = pd.DataFrame({
    "column": ["problemas"] * len(ISSUE_OPTIONS) + ["ajustes"] * len(FIX_OPTIONS),
    "code": ISSUE_OPTIONS + FIX_OPTIONS,
})
summary["rows"] = [int(codes[c].sum()) for c in summary.code]
summary["persons"] = [int(person_codes[c].sum()) for c in summary.code]
print(f"{len(rows):,} shipped trip rows, {len(persons):,} persons with trips; notes file: {NOTES_PATH}")
summary.style.hide(axis="index")

In [ ]:
DISPLAY_DROP = (review.KEYS[:2] + review.PERSON_COLUMNS + [review.NOTE]
                + [review.new_column(c) for c in review.EDITABLE])      # the new-value columns are for the CSV


def person_frame(key, matching, source=None, hab=None):
    # the person's rows as the browser shows them: review.chain_sheet, minus the columns the header carries
    t = review.chain_sheet(rows if source is None else source, cleaned.hab if hab is None else hab, [key])
    t = t.drop(columns=DISPLAY_DROP)
    t["_match"] = [bool(matching.get((key[0], key[1], fv), False)) for fv in t.trip]
    return t


def style_frame(df):
    def row_css(r):
        if r["status"] == "dropped":
            return ["color:#999; text-decoration:line-through"] * len(r)
        if r["_match"]:
            return ["background-color:#fff3b0"] * len(r)
        return [""] * len(r)
    return (df.style.apply(row_css, axis=1).hide(axis="index").hide(axis="columns", subset=["_match"])
            .set_table_styles([{"selector": "th, td", "props": "padding:2px 8px; font-size:12px; text-align:left"}]))


def header_html(key):
    p = persons.loc[key]
    codes_txt = ", ".join(sorted(p.issue_codes | p.fix_codes | ({"dropped"} if p.dropped else set()))) or "clean"
    rep = " · <b>repeated diary</b>" if p.diario_repetido else ""
    return (f"<h4 style='margin:4px 0'>household {key[0]}, person {key[1]}</h4>"
            f"<div style='font-size:12px'>home zone {p.home} · {p.sexo_nacimiento}, {int(p.edad)} · {p.ocupacion if pd.notna(p.ocupacion) else '—'} · "
            f"{int(p.viajes_contados)} trips counted, {int(p.n_trips)} rows shipped{rep}<br>codes: {codes_txt}</div>")


def review_html(key):
    v = review_state["verdict"].persons.set_index(["household", "person"]).loc[key]
    what = ", ".join(f"{int(v[c])} {c}" for c in ("cells", "dropped", "restored", "notes") if v[c])
    return (f"<div style='font-size:12px; margin-top:6px'><b>after your edits</b> ({what}): "
            f"problemas {v.before or 'none'} → <b>{v.after or 'none'}</b></div>")


def timeline(key):
    lanes = [("after load_eod", rows, "clean"), ("as shipped", rows, "shipped")]
    if key in review_state["persons"]:
        lanes.insert(0, ("after your edits", review_state["rows"], "clean"))
    fig, ax = plt.subplots(figsize=(10, 1.6 + 0.7 * len(lanes)))
    for lane, (label, source, side) in enumerate(lanes):
        for fv, r in source.loc[key].iterrows():
            s = r[f"start_{side}"]
            if pd.isna(s):
                continue
            color = "#969696" if r[f"motive_{side}"] == HOME_MOTIVE else "#2c7fb8"
            flagged = side == "clean" and bool(r.problemas)
            edited = side == "clean" and bool(r.ajustes)
            ax.broken_barh([(s / 60, max(int(r.leg_min), 4) / 60)], (lane - 0.35, 0.7), facecolors=color,
                           edgecolors="#e6550d" if flagged else color, linewidth=1.8, hatch="///" if edited else None)
            ax.text(s / 60, lane + 0.38, str(fv), fontsize=8, ha="left", va="bottom")
    ax.set_yticks(range(len(lanes)))
    ax.set_yticklabels([label for label, *_ in lanes])
    ax.set_ylim(-0.7, len(lanes) - 0.1)
    ax.set_xlim(0, 24)
    ax.set_xticks(range(0, 25, 2))
    ax.set_xlabel("hour of day")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()


# ---- widgets
issue_sel = w.SelectMultiple(options=ISSUE_OPTIONS, rows=9, description="problemas", layout=w.Layout(width="320px"))
fix_sel = w.SelectMultiple(options=FIX_OPTIONS, rows=9, description="ajustes", layout=w.Layout(width="320px"))
mode_sel = w.ToggleButtons(options=["any selected", "all selected"], value="any selected", description="match")
min_trips = w.IntSlider(min=1, max=int(persons.n_trips.max()), value=1, description="min trips", continuous_update=False)
diary_sel = w.Dropdown(options=[("any diary", None), ("repeated diary", True), ("not repeated", False)], value=None,
                       description="diary")
noted_only = w.Checkbox(value=False, description="with a note only", indent=False)
edited_only = w.Checkbox(value=False, description="edited only", indent=False)
prev_btn, next_btn, rand_btn = w.Button(description="◀"), w.Button(description="▶"), w.Button(description="random")
pos = w.BoundedIntText(min=1, max=1, value=1, description="#", layout=w.Layout(width="160px"))
count_lbl = w.HTML()
goto = w.Text(placeholder="household,person", description="go to", layout=w.Layout(width="260px"))
note_box = w.Textarea(placeholder="a note about this person's chain…", layout=w.Layout(width="640px", height="60px"))
save_btn = w.Button(description="save note")
note_lbl = w.HTML()
review_path = w.Text(value=str(REVIEW_PATH) if REVIEW_PATH.exists() else "", placeholder="path to an edited review sheet",
                     description="review sheet", layout=w.Layout(width="520px"))
load_btn = w.Button(description="load sheet")
review_lbl = w.HTML()
out = w.Output()

state = {"keys": [], "i": 0, "matching": pd.Series(False, index=rows.index), "busy": False}
review_state = {"rows": None, "hab": None, "persons": set(), "verdict": None}


def selected_codes():
    return list(issue_sel.value), list(fix_sel.value)


def filtered_keys():
    issues, fixes = selected_codes()
    chosen = issues + fixes
    if chosen:
        matching = codes[chosen].any(axis=1)
        keep = person_codes[chosen].all(axis=1) if mode_sel.value == "all selected" else person_codes[chosen].any(axis=1)
    else:
        matching = pd.Series(False, index=rows.index)
        keep = pd.Series(True, index=persons.index)
    keep &= persons.n_trips >= min_trips.value
    if diary_sel.value is not None:
        keep &= persons.diario_repetido == diary_sel.value
    if noted_only.value:
        keep &= pd.Series([f"{k[0]}/{k[1]}" in notes for k in persons.index], index=persons.index)
    if edited_only.value:
        keep &= pd.Series([tuple(k) in review_state["persons"] for k in persons.index], index=persons.index)
    return [tuple(k) for k in persons.index[keep]], matching


def render():
    keys = state["keys"]
    with out:
        out.clear_output(wait=True)
        if not keys:
            display(HTML("<i>no person matches the selection</i>"))
            return
        key = keys[state["i"]]
        display(HTML(header_html(key)))
        display(style_frame(person_frame(key, state["matching"])))
        if key in review_state["persons"]:
            display(HTML(review_html(key)))
            display(style_frame(person_frame(key, state["matching"], review_state["rows"], review_state["hab"])))
        timeline(key)
    note_box.value = notes.get(f"{key[0]}/{key[1]}", "") if keys else ""
    count_lbl.value = f"<b>{state['i'] + 1:,} / {len(keys):,}</b> persons" if keys else "<b>0</b> persons"


def move_to(i):
    if not state["keys"]:
        return
    state["i"] = int(i) % len(state["keys"])
    state["busy"] = True
    pos.value = state["i"] + 1
    state["busy"] = False
    render()


def refresh(_=None):
    current = state["keys"][state["i"]] if state["keys"] else None
    state["keys"], state["matching"] = filtered_keys()
    pos.max = max(len(state["keys"]), 1)
    move_to(state["keys"].index(current) if current in state["keys"] else 0)


def on_pos(change):
    if not state["busy"]:
        move_to(change["new"] - 1)


def on_goto(_):
    try:
        hh, person = (int(x) for x in goto.value.replace(" ", "").split(","))
    except ValueError:
        return
    key = (hh, person)
    if key not in persons.index:
        count_lbl.value = f"<i>no trips for household {hh}, person {person}</i>"
        return
    if key not in state["keys"]:
        state["keys"] = [key]
        pos.max = 1
    move_to(state["keys"].index(key))


def on_save(_):
    if not state["keys"]:
        return
    key = state["keys"][state["i"]]
    k = f"{key[0]}/{key[1]}"
    if note_box.value.strip():
        notes[k] = note_box.value.strip()
    else:
        notes.pop(k, None)
    NOTES_PATH.write_text(json.dumps(notes, indent=2, ensure_ascii=False))
    note_lbl.value = f"<span style='font-size:12px'>{len(notes)} notes saved to {NOTES_PATH.name}</span>"


def load_review(_=None):
    # the filled new-value cells of the sheet are the edits: apply them to the live tables, recompute problemas
    path = Path(review_path.value.strip()).expanduser()
    if not path.is_file():
        review_lbl.value = f"<span style='font-size:12px; color:#b00'>no file at {path}</span>"
        return
    try:
        edits = review.sheet_edits(review.read_sheet(path))
        if edits.empty:
            review_state.update(rows=None, hab=None, persons=set(), verdict=None)
            review_lbl.value = f"<span style='font-size:12px'>{path.name}: no edits yet</span>"
            refresh()
            return
        verified = review.verify_edits(cleaned, shipped, edits)
    except ValueError as err:
        review_lbl.value = f"<pre style='font-size:12px; color:#b00; margin:0'>{err}</pre>"
        return
    review_state.update(rows=review.chain_rows(verified.tables, shipped), hab=verified.tables.hab, verdict=verified,
                        persons={(int(h), int(p)) for h, p in verified.persons[["household", "person"]].itertuples(index=False)})
    s = verified.summary
    review_lbl.value = (f"<span style='font-size:12px'>{path.name}: {s['persons']} persons edited, {s['cells']} cells, "
                        f"{s['rows_dropped']} rows dropped, {s['rows_restored']} restored · rows with a defect "
                        f"{s['defects_before']} → {s['defects_after']}, {s['persons_cleared']} persons cleared, "
                        f"{s['persons_with_new_defects']} with a new defect</span>")
    refresh()


for widget in (issue_sel, fix_sel, mode_sel, min_trips, diary_sel, noted_only, edited_only):
    widget.observe(refresh, "value")
pos.observe(on_pos, "value")
prev_btn.on_click(lambda _: move_to(state["i"] - 1))
next_btn.on_click(lambda _: move_to(state["i"] + 1))
rand_btn.on_click(lambda _: move_to(random.randrange(len(state["keys"]))) if state["keys"] else None)
goto.continuous_update = False          # fires on Enter or focus loss
goto.observe(on_goto, "value")
save_btn.on_click(on_save)
load_btn.on_click(load_review)

ui = w.VBox([
    w.HBox([issue_sel, fix_sel, w.VBox([mode_sel, min_trips, diary_sel, noted_only, edited_only])]),
    w.HBox([review_path, load_btn, review_lbl]),
    w.HBox([prev_btn, next_btn, pos, count_lbl, rand_btn, goto]),
    out,
    w.HBox([note_box, w.VBox([save_btn, note_lbl])]),
])
if REVIEW_PATH.exists():
    load_review()
refresh()
ui

## Starting points

- `hora_invertida` with `hora:vecinos` under *all selected*: the imputed rows the typo search could not
  rescue, to check that the fault sits in the neighbours' times.
- `origen_discontinuo` alone: the 32 zone breaks left, to look for which side is wrong.
- `regreso_sin_llegar` against `tipo_destino_dudoso`: returns that did not reach home versus returns that
  did with a doubtful type, the two sides of the recode's zone guard.
- `hora:-10h+12h`: the compound two-hour reading, the most expensive edit in the menu, to validate it holds
  up chain by chain.
- `inicio_fuera_de_casa` with `fin_fuera_de_casa`: days away from home at both ends, night shifts and stays
  elsewhere.
- *edited only* after loading a review sheet: the persons whose chain was edited by hand, each with the
  defects the edits cleared, left or made; `eodgdl review verify notebooks/chain_review.csv --data data`
  prints the same verdict per person.